In [5]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [6]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по птице v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Птица
211,АКТЮБИНСКАЯ ОБЛАСТЬ,2022-01-01,103.40
909,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2023-01-01,1545.70
1525,ОБЛАСТЬ АБАЙ,2022-08-01,2256.80
92,АКМОЛИНСКАЯ ОБЛАСТЬ,2022-09-01,8804.75
1308,КОСТАНАЙСКАЯ ОБЛАСТЬ,2024-07-01,545.87
376,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-03-01,12752.96
1329,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2015-09-01,0.66
1894,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2019-09-01,746.52
502,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2020-01-01,4653.89
153,АКТЮБИНСКАЯ ОБЛАСТЬ,2017-03-01,323.90


In [7]:
regions = df['Регион'].unique()
target   = "Птица"
horizon  = 3
epsilon = 1e-6

In [8]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [9]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Птица - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Птица - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[11268.8, 10962.08, 10960.74]","[10080.07, 10273.59, 10251.41]",892.61,862.18,8.47
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[10252.48, 10257.31, 10525.67]","[10273.59, 10251.41, 8665.21]",1074.21,629.16,7.24
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[10269.94, 10538.61, 11740.42]","[10251.41, 8665.21, 11438.74]",1095.59,731.20,8.15
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[10527.16, 11727.89, 9571.69]","[8665.21, 11438.74, 8560.3]",1234.69,1054.16,11.94
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[10434.02, 8525.1, 8451.39]","[11438.74, 8560.3, 12551.51]",2437.33,1713.35,13.95
...,...,...,...,...,...,...,...,...
168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[354.92, 526.86, 541.42]","[810.6, 641.64, 802.3]",310.31,277.11,35.54
169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[751.03, 767.97, 1103.05]","[641.64, 802.3, 750.07]",214.27,165.57,22.80
170,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[717.59, 1031.78, 758.58]","[802.3, 750.07, 754.31]",169.86,123.56,16.23
171,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[1081.73, 794.83, 942.02]","[750.07, 754.31, 925.67]",193.14,129.51,17.12


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,1670.51,1445.79,13.02
1,АКТЮБИНСКАЯ ОБЛАСТЬ,34.03,26.98,277.84
2,АЛМАТИНСКАЯ ОБЛАСТЬ,1183.86,1044.20,8.81
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,569.14,522.48,8.55
4,ГАЛМАТЫ,0.11,0.11,NaN
5,ГАСТАНА,0.08,0.07,NaN
6,ГШЫМКЕНТ,8.77,7.97,168.34
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,584.18,511.77,25.54
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,265.84,214.58,20.69
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,96.68,80.45,8.14


In [10]:
# res_hw[res_hw['Регион'] == 'АКТЮБИНСКАЯ ОБЛАСТЬ']

## SARIMA

In [ ]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Птица - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Птица - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

## Facebook Prophet

In [ ]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Птица - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Птица - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


In [ ]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Птица - Лучшие модели v2.xlsx", index=False)


In [ ]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Птица - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Птица - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Птица - Результаты прогнозов Prophet средние v2.xlsx"


# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [ ]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Птица - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result